# Guide v2 · Train6 · Đọc hết câu

Gắn hai Dataset riêng tư **Guide v2 Models** và **Guide v2 Train6**, chọn GPU T4 và bật Internet, rồi Run All. Đường dẫn và thời lượng đã được điền sẵn.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['YOLO_AUTOINSTALL'] = 'false'
from pathlib import Path
print('Các thư mục input:')
for p in Path('/kaggle/input').iterdir(): print(p)
print('Các manifest, ZIP và video trong Input:')
for pattern in ('bundle_manifest.json', 'guide_kaggle_v2*.zip', '*.mp4', '*.mov', '*.MOV'):
    for p in Path('/kaggle/input').rglob(pattern): print(p)


## 1. Chọn gói đã tải lên

Đặt `BUNDLE_SOURCE` thành đường dẫn file ZIP hoặc thư mục chứa `bundle_manifest.json` được liệt kê ở trên. Chọn video ở bước 3.


In [ ]:
BUNDLE_SOURCE = Path('/kaggle/input/datasets/nguyenduchng/guide-v2-models')
WORK = Path('/kaggle/working/guide_v2')
import zipfile, shutil, json, hashlib
assert BUNDLE_SOURCE.exists(), 'Sửa BUNDLE_SOURCE theo danh sách ở ô trên.'
WORK.mkdir(parents=True, exist_ok=True)
if BUNDLE_SOURCE.is_file():
    with zipfile.ZipFile(BUNDLE_SOURCE) as z:
        for name in z.namelist():
            dest = (WORK / name).resolve()
            assert dest.is_relative_to(WORK.resolve()) and '\\' not in name, 'Unsafe ZIP path'
        z.extractall(WORK)
else:
    manifest = json.loads((BUNDLE_SOURCE/'bundle_manifest.json').read_text())
    for item in manifest['files']:
        name = item['path']
        dest = (WORK/name).resolve()
        assert dest.is_relative_to(WORK.resolve()) and '\\' not in name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(BUNDLE_SOURCE/name, dest)
    shutil.copy2(BUNDLE_SOURCE/'bundle_manifest.json', WORK/'bundle_manifest.json')
manifest = json.loads((WORK/'bundle_manifest.json').read_text())
for item in manifest['files']:
    p = (WORK/item['path']).resolve()
    assert p.is_relative_to(WORK.resolve())
    h = hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''): h.update(chunk)
    assert h.hexdigest() == item['sha256'], f'File lỗi: {p}'
os.chdir(WORK)
print('Gói kiểm tra toàn vẹn OK; video có trong gói:', manifest.get('video'))

# Hai nguồn đi kèm: trọng số ở Guide v2 Models, mã cập nhật trong ô này.
from base64 import b64decode
speech_update = {
    'guide_core/audio.py': (
        'IiIiRGV0ZXJtaW5pc3RpYyBzaW5nbGUtdm9pY2Ugc2NoZWR1bGVyLiBPZmZsaW5lICdzY2hlZHVsZWQnIGlzIE5PVCBzcGVha2Vy'
        'IGRlbGl2ZXJ5LiIiIgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHdhdmUKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20g'
        'LnR5cGVzIGltcG9ydCBBY3Rpb24sIERJUkVDVElPTkFMLCBTSURFX0dVSURBTkNFLCBTUEVFQ0hfUFJJT1JJVFkKCkNMSVBTPXth'
        'OmEudmFsdWUubG93ZXIoKSsiLndhdiIgZm9yIGEgaW4gQWN0aW9ufQoKY2xhc3MgQ2xpcEJhbms6CiAgICBkZWYgX19pbml0X18o'
        'c2VsZixkaXJlY3Rvcnksc2FtcGxlX3JhdGU9MjQwMDApOgogICAgICAgIHNlbGYucmF0ZT1zYW1wbGVfcmF0ZQogICAgICAgIHNl'
        'bGYuY2xpcHM9e30KICAgICAgICBmb3IgYWN0aW9uLG5hbWUgaW4gQ0xJUFMuaXRlbXMoKToKICAgICAgICAgICAgd2l0aCB3YXZl'
        'Lm9wZW4oc3RyKFBhdGgoZGlyZWN0b3J5KS9uYW1lKSwicmIiKSBhcyBmOgogICAgICAgICAgICAgICAgaWYgZi5nZXRzYW1wd2lk'
        'dGgoKSE9MjogcmFpc2UgVmFsdWVFcnJvcigiVm9pY2UgY2xpcHMgbXVzdCBiZSBQQ00xNiBXQVYuIikKICAgICAgICAgICAgICAg'
        'IGF1ZGlvPW5wLmZyb21idWZmZXIoZi5yZWFkZnJhbWVzKGYuZ2V0bmZyYW1lcygpKSxkdHlwZT0iPGkyIikuYXN0eXBlKG5wLmZs'
        'b2F0MzIpCiAgICAgICAgICAgICAgICBhdWRpbz1hdWRpby5yZXNoYXBlKC0xLGYuZ2V0bmNoYW5uZWxzKCkpLm1lYW4oMSkKICAg'
        'ICAgICAgICAgICAgIGlmIGYuZ2V0ZnJhbWVyYXRlKCkhPXNlbGYucmF0ZToKICAgICAgICAgICAgICAgICAgICBhdWRpbz1ucC5p'
        'bnRlcnAobnAuYXJhbmdlKHJvdW5kKGxlbihhdWRpbykqc2VsZi5yYXRlL2YuZ2V0ZnJhbWVyYXRlKCkpKSpmLmdldGZyYW1lcmF0'
        'ZSgpL3NlbGYucmF0ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbnAuYXJhbmdlKGxlbihhdWRpbykpLGF1'
        'ZGlvKQogICAgICAgICAgICAgICAgc2VsZi5jbGlwc1thY3Rpb25dPWF1ZGlvLmFzdHlwZSgiPGkyIikKICAgICAgICBzZWxmLmV4'
        'dHJhPXt9CiAgICAgICAgcGF0aD1QYXRoKGRpcmVjdG9yeSkvIm5hcnJvdy53YXYiCiAgICAgICAgaWYgcGF0aC5pc19maWxlKCk6'
        'CiAgICAgICAgICAgIHdpdGggd2F2ZS5vcGVuKHN0cihwYXRoKSwicmIiKSBhcyBmOgogICAgICAgICAgICAgICAgaWYgZi5nZXRz'
        'YW1wd2lkdGgoKSE9MiBvciBmLmdldGZyYW1lcmF0ZSgpIT1zZWxmLnJhdGUgb3IgZi5nZXRuY2hhbm5lbHMoKSE9MToKICAgICAg'
        'ICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJuYXJyb3cud2F2IG11c3QgYmUgbW9ubyBQQ00xNiBhdCBiYW5rIHNhbXBs'
        'ZSByYXRlIikKICAgICAgICAgICAgICAgIHNlbGYuZXh0cmFbIm5hcnJvd19jb3JyaWRvciJdPW5wLmZyb21idWZmZXIoZi5yZWFk'
        'ZnJhbWVzKGYuZ2V0bmZyYW1lcygpKSxkdHlwZT0iPGkyIikKCiAgICBkZWYgc2FtcGxlcyhzZWxmLGRlY2lzaW9uKToKICAgICAg'
        'ICByZXR1cm4gc2VsZi5leHRyYS5nZXQoZGVjaXNpb24ucmVhc29uLHNlbGYuY2xpcHNbZGVjaXNpb24uYWN0aW9uXSkKCmNsYXNz'
        'IEF1ZGlvVGltZWxpbmU6CiAgICBkZWYgX19pbml0X18oc2VsZixiYW5rLHJlcGVhdF9zPTQuLHNpbms9Tm9uZSk6CiAgICAgICAg'
        'c2VsZi5iYW5rPWJhbmsKICAgICAgICBzZWxmLnJlcGVhdF9zPXJlcGVhdF9zCiAgICAgICAgc2VsZi5zaW5rPXNpbmsgIyBPcHRp'
        'b25hbCBzdWJtaXQoc2FtcGxlcywgcmF0ZSkgLyBzdG9wKCk7IG9ubHkgbGl2ZSBzaW5rcyBsb2cgYWN0dWFsIHN1Ym1pc3Npb24u'
        'CiAgICAgICAgc2VsZi5hY3RpdmU9Tm9uZQogICAgICAgIHNlbGYucGVuZGluZz1Ob25lCiAgICAgICAgc2VsZi5wZW5kaW5nX3Np'
        'bmNlPU5vbmUKICAgICAgICBzZWxmLmxhdGVzdD1Ob25lCiAgICAgICAgc2VsZi5zZWdtZW50cz1bXQogICAgICAgIHNlbGYuZXZl'
        'bnRzPVtdCiAgICAgICAgc2VsZi5sYXN0X3N0YXJ0ZWQ9e30KICAgICAgICBzZWxmLmxhc3Rfc2lkZV9zdGFydGVkPU5vbmUKICAg'
        'ICAgICBzZWxmLmVuYWJsZWQ9VHJ1ZQoKICAgIGRlZiBfbG9nKHNlbGYsZXZlbnQsbm93LGRlY2lzaW9uLCoqZXh0cmEpOgogICAg'
        'ICAgIHNlbGYuZXZlbnRzLmFwcGVuZChkaWN0KGV2ZW50PWV2ZW50LHRpbWVfcz1ub3csYWN0aW9uPWRlY2lzaW9uLmFjdGlvbi52'
        'YWx1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBldmlkZW5jZT1kZWNpc2lvbi5yZWNvcmQoKSwqKmV4dHJhKSkK'
        'CiAgICBkZWYgX2NhbmNlbChzZWxmLG5vdyx3aHkpOgogICAgICAgIGlmIHNlbGYuYWN0aXZlOgogICAgICAgICAgICBzZWdtZW50'
        'PXNlbGYuYWN0aXZlCiAgICAgICAgICAgIHNlZ21lbnRbImVuZCJdPW1pbihzZWdtZW50WyJlbmQiXSxub3cpCiAgICAgICAgICAg'
        'IHNlbGYuX2xvZygiY2FuY2VsbGVkIixub3csc2VnbWVudFsiZGVjaXNpb24iXSxyZWFzb249d2h5KQogICAgICAgICAgICBpZiBz'
        'ZWxmLnNpbms6IHNlbGYuc2luay5zdG9wKCkKICAgICAgICAgICAgc2VsZi5hY3RpdmU9Tm9uZQoKICAgIGRlZiBzdG9wKHNlbGYs'
        'bm93KToKICAgICAgICBzZWxmLl9jYW5jZWwobm93LCJzdG9wcGVkX29yX211dGVkIikKICAgICAgICBzZWxmLnBlbmRpbmc9Tm9u'
        'ZQogICAgICAgIHNlbGYucGVuZGluZ19zaW5jZT1Ob25lCiAgICAgICAgc2VsZi5lbmFibGVkPUZhbHNlCgogICAgZGVmIF9maW5p'
        'c2hfY29tcGxldGVkKHNlbGYsIG5vdyk6CiAgICAgICAgaWYgc2VsZi5hY3RpdmUgaXMgTm9uZSBvciBub3cgPCBzZWxmLmFjdGl2'
        'ZVsiZW5kIl06CiAgICAgICAgICAgIHJldHVybgogICAgICAgIHNlbGYuX2xvZygiZmluaXNoZWQiLCBzZWxmLmFjdGl2ZVsiZW5k'
        'Il0sIHNlbGYuYWN0aXZlWyJkZWNpc2lvbiJdKQogICAgICAgIHNlbGYuYWN0aXZlID0gTm9uZQoKICAgIGRlZiBfcXVldWUoc2Vs'
        'ZiwgZGVjaXNpb24sIG5vdyk6CiAgICAgICAgaWYgc2VsZi5wZW5kaW5nIGlzIE5vbmUgb3Igc2VsZi5wZW5kaW5nLmFjdGlvbiAh'
        'PSBkZWNpc2lvbi5hY3Rpb246CiAgICAgICAgICAgIHNlbGYucGVuZGluZ19zaW5jZSA9IG5vdwogICAgICAgIHNlbGYucGVuZGlu'
        'ZyA9IGRlY2lzaW9uCgogICAgZGVmIF9jbGVhcl9wZW5kaW5nKHNlbGYpOgogICAgICAgIHNlbGYucGVuZGluZyA9IE5vbmUKICAg'
        'ICAgICBzZWxmLnBlbmRpbmdfc2luY2UgPSBOb25lCgogICAgZGVmIF9jYW5fc3RhcnQoc2VsZiwgZGVjaXNpb24sIG5vdyk6CiAg'
        'ICAgICAgYWN0aW9uID0gZGVjaXNpb24uYWN0aW9uCiAgICAgICAgaWYgYWN0aW9uIGluIFNJREVfR1VJREFOQ0UgYW5kIGFjdGlv'
        'biAhPSBzZWxmLmxhc3Rfc2lkZV9zdGFydGVkOgogICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgIHJldHVybiBub3cgLSBz'
        'ZWxmLmxhc3Rfc3RhcnRlZC5nZXQoYWN0aW9uLCBmbG9hdCgiLWluZiIpKSA+PSBzZWxmLnJlcGVhdF9zCgogICAgZGVmIF9zdGFy'
        'dChzZWxmLGRlY2lzaW9uLG5vdyxxdWV1ZWRfYXQ9Tm9uZSk6CiAgICAgICAgaWYgbm90IHNlbGYuZW5hYmxlZCBvciBkZWNpc2lv'
        'bi52YWxpZF91bnRpbDxub3c6CiAgICAgICAgICAgIHNlbGYuX2xvZygiZXhwaXJlZCIsbm93LGRlY2lzaW9uKQogICAgICAgICAg'
        'ICByZXR1cm4KICAgICAgICBzYW1wbGVzPXNlbGYuYmFuay5zYW1wbGVzKGRlY2lzaW9uKQogICAgICAgIHNlbGYuYWN0aXZlPXsi'
        'c3RhcnQiOm5vdywiZW5kIjpub3crbGVuKHNhbXBsZXMpL3NlbGYuYmFuay5yYXRlLCJkZWNpc2lvbiI6ZGVjaXNpb24sInNhbXBs'
        'ZXMiOnNhbXBsZXN9CiAgICAgICAgc2VsZi5zZWdtZW50cy5hcHBlbmQoc2VsZi5hY3RpdmUpCiAgICAgICAgc2VsZi5sYXN0X3N0'
        'YXJ0ZWRbZGVjaXNpb24uYWN0aW9uXT1ub3cKICAgICAgICBpZiBkZWNpc2lvbi5hY3Rpb24gaW4gU0lERV9HVUlEQU5DRToKICAg'
        'ICAgICAgICAgc2VsZi5sYXN0X3NpZGVfc3RhcnRlZCA9IGRlY2lzaW9uLmFjdGlvbgogICAgICAgIGVsaWYgZGVjaXNpb24uYWN0'
        'aW9uID09IEFjdGlvbi5TVE9QOgogICAgICAgICAgICBzZWxmLmxhc3Rfc2lkZV9zdGFydGVkID0gTm9uZQogICAgICAgIGV2ZW50'
        'PSJzY2hlZHVsZWQiCiAgICAgICAgc3VibWlzc2lvbl9tcz1Ob25lCiAgICAgICAgaWYgc2VsZi5zaW5rOgogICAgICAgICAgICBp'
        'bXBvcnQgdGltZQogICAgICAgICAgICB0PXRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICAgICAgc2VsZi5zaW5rLnN1Ym1pdChz'
        'YW1wbGVzLHNlbGYuYmFuay5yYXRlKQogICAgICAgICAgICBzdWJtaXNzaW9uX21zPSh0aW1lLnBlcmZfY291bnRlcigpLXQpKjEw'
        'MDAKICAgICAgICAgICAgZXZlbnQ9InN1Ym1pdHRlZF90b19kZXZpY2UiCiAgICAgICAgc2VsZi5fbG9nKGV2ZW50LG5vdyxkZWNp'
        'c2lvbixxdWV1ZV93YWl0X3M9bWF4KDAsbm93LShub3cgaWYgcXVldWVkX2F0IGlzIE5vbmUgZWxzZSBxdWV1ZWRfYXQpKSwKICAg'
        'ICAgICAgICAgICAgICAgc3VibWlzc2lvbl9tcz1zdWJtaXNzaW9uX21zKQoKICAgIGRlZiB1cGRhdGUoc2VsZiwgZGVjaXNpb24s'
        'IG5vdyk6CiAgICAgICAgaWYgbm90IHNlbGYuZW5hYmxlZDoKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgaWYgZGVjaXNpb24u'
        'dmFsaWRfdW50aWwgPCBub3c6CiAgICAgICAgICAgIHNlbGYuX2xvZygiZXhwaXJlZCIsIG5vdywgZGVjaXNpb24pCiAgICAgICAg'
        'ICAgIHNlbGYudGljayhub3cpCiAgICAgICAgICAgIHJldHVybgoKICAgICAgICBzZWxmLl9maW5pc2hfY29tcGxldGVkKG5vdykK'
        'ICAgICAgICBwcmV2aW91cyA9IHNlbGYubGF0ZXN0CiAgICAgICAgaWYgZGVjaXNpb24uYWN0aW9uID09IEFjdGlvbi5TVE9QIGFu'
        'ZCAocHJldmlvdXMgaXMgTm9uZSBvciBwcmV2aW91cy5hY3Rpb24gIT0gQWN0aW9uLlNUT1ApOgogICAgICAgICAgICAjIENvYWxl'
        'c2NlIG9ic2VydmF0aW9ucyBvZiB0aGUgc2FtZSBhbGFybS4gQSBuZXcgYWxhcm0gaXMgZWxpZ2libGUgaW1tZWRpYXRlbHkuCiAg'
        'ICAgICAgICAgIHNlbGYubGFzdF9zdGFydGVkLnBvcChBY3Rpb24uU1RPUCwgTm9uZSkKICAgICAgICBpZiBwcmV2aW91cyBpcyBO'
        'b25lIG9yIHByZXZpb3VzLmFjdGlvbiAhPSBkZWNpc2lvbi5hY3Rpb246CiAgICAgICAgICAgIHNlbGYuX2xvZygicmVxdWVzdGVk'
        'Iiwgbm93LCBkZWNpc2lvbikKICAgICAgICBzZWxmLmxhdGVzdCA9IGRlY2lzaW9uCgogICAgICAgIGlmIHNlbGYucGVuZGluZzoK'
        'ICAgICAgICAgICAgcGVuZGluZyA9IHNlbGYucGVuZGluZwogICAgICAgICAgICBleHBpcmVkID0gcGVuZGluZy52YWxpZF91bnRp'
        'bCA8IG5vdwogICAgICAgICAgICByZXZva2VkID0gcGVuZGluZy5mcmFtZV9rZXlbMF0gIT0gZGVjaXNpb24uZnJhbWVfa2V5WzBd'
        'IG9yICgKICAgICAgICAgICAgICAgIHBlbmRpbmcuYWN0aW9uIGluIERJUkVDVElPTkFMCiAgICAgICAgICAgICAgICBhbmQgKHBl'
        'bmRpbmcuYWN0aW9uICE9IGRlY2lzaW9uLmFjdGlvbiBvciBkZWNpc2lvbi5pbnZhbGlkYXRlc19yb3V0ZSkKICAgICAgICAgICAg'
        'KQogICAgICAgICAgICBpZiBleHBpcmVkIG9yIHJldm9rZWQ6CiAgICAgICAgICAgICAgICBzZWxmLl9sb2coImV4cGlyZWQiIGlm'
        'IGV4cGlyZWQgZWxzZSAicGVuZGluZ19yZXBsYWNlZCIsIG5vdywgcGVuZGluZykKICAgICAgICAgICAgICAgIHNlbGYuX2NsZWFy'
        'X3BlbmRpbmcoKQogICAgICAgICAgICBlbGlmIFNQRUVDSF9QUklPUklUWS5nZXQocGVuZGluZy5hY3Rpb24sIDEpID4gU1BFRUNI'
        'X1BSSU9SSVRZLmdldChkZWNpc2lvbi5hY3Rpb24sIDEpOgogICAgICAgICAgICAgICAgIyBBIHRyYW5zaWVudCByb3V0aW5lIHVw'
        'ZGF0ZSBjYW5ub3QgZGlzcGxhY2UgYSBzdGlsbC1mcmVzaCBTVE9QLgogICAgICAgICAgICAgICAgc2VsZi50aWNrKG5vdykKICAg'
        'ICAgICAgICAgICAgIHJldHVybgogICAgICAgICAgICBlbGlmIHBlbmRpbmcuYWN0aW9uICE9IGRlY2lzaW9uLmFjdGlvbjoKICAg'
        'ICAgICAgICAgICAgIHNlbGYuX2xvZygicGVuZGluZ19yZXBsYWNlZCIsIG5vdywgcGVuZGluZykKICAgICAgICAgICAgICAgIHNl'
        'bGYuX2NsZWFyX3BlbmRpbmcoKQogICAgICAgIGlmIHNlbGYuYWN0aXZlIGlzIE5vbmUgb3Igc2VsZi5hY3RpdmVbImRlY2lzaW9u'
        'Il0uYWN0aW9uICE9IGRlY2lzaW9uLmFjdGlvbjoKICAgICAgICAgICAgc2VsZi5fcXVldWUoZGVjaXNpb24sIG5vdykKICAgICAg'
        'ICBzZWxmLnRpY2sobm93KQoKICAgIGRlZiB0aWNrKHNlbGYsIG5vdyk6CiAgICAgICAgc2VsZi5fZmluaXNoX2NvbXBsZXRlZChu'
        'b3cpCiAgICAgICAgaWYgc2VsZi5wZW5kaW5nIGFuZCBub3cgPiBzZWxmLnBlbmRpbmcudmFsaWRfdW50aWw6CiAgICAgICAgICAg'
        'IHNlbGYuX2xvZygiZXhwaXJlZCIsIG5vdywgc2VsZi5wZW5kaW5nKQogICAgICAgICAgICBzZWxmLl9jbGVhcl9wZW5kaW5nKCkK'
        'ICAgICAgICBpZiBub3Qgc2VsZi5hY3RpdmUgYW5kIHNlbGYucGVuZGluZyBhbmQgc2VsZi5fY2FuX3N0YXJ0KHNlbGYucGVuZGlu'
        'Zywgbm93KToKICAgICAgICAgICAgZGVjaXNpb24sIHF1ZXVlZF9hdCA9IHNlbGYucGVuZGluZywgc2VsZi5wZW5kaW5nX3NpbmNl'
        'CiAgICAgICAgICAgIHNlbGYuX2NsZWFyX3BlbmRpbmcoKQogICAgICAgICAgICBzZWxmLl9zdGFydChkZWNpc2lvbiwgbm93LCBx'
        'dWV1ZWRfYXQpCgogICAgZGVmIGZpbmlzaF9leHBvcnQoc2VsZiwgdmlkZW9fZW5kKToKICAgICAgICAiIiJDbG9zZSB0aGUgY3Vy'
        'cmVudCBwaHJhc2UgYW5kIGRyb3AgcXVldWVkIGNvbW1hbmRzIGF0IGVuZCBvZiBmb290YWdlLiIiIgogICAgICAgIHNlbGYuX2Ns'
        'ZWFyX3BlbmRpbmcoKQogICAgICAgIGVuZCA9IG1heCh2aWRlb19lbmQsIHNlbGYuYWN0aXZlWyJlbmQiXSkgaWYgc2VsZi5hY3Rp'
        'dmUgZWxzZSB2aWRlb19lbmQKICAgICAgICBzZWxmLl9maW5pc2hfY29tcGxldGVkKGVuZCkKICAgICAgICBzZWxmLmVuYWJsZWQg'
        'PSBGYWxzZQogICAgICAgIHJldHVybiBlbmQKCiAgICBkZWYgc2F2ZShzZWxmLHBhdGgsZHVyYXRpb24pOgogICAgICAgICMgU3Ry'
        'ZWFtIGNodW5rcyB0byBkaXNrIGluc3RlYWQgb2YgYWxsb2NhdGluZyBhIGZ1bGwgbG9uZy12aWRlbyBXQVYgaW4gUkFNLgogICAg'
        'ICAgIHdpdGggd2F2ZS5vcGVuKHN0cihwYXRoKSwid2IiKSBhcyBvdXQ6CiAgICAgICAgICAgIG91dC5zZXRuY2hhbm5lbHMoMSk7'
        'b3V0LnNldHNhbXB3aWR0aCgyKTtvdXQuc2V0ZnJhbWVyYXRlKHNlbGYuYmFuay5yYXRlKQogICAgICAgICAgICB0b3RhbD1yb3Vu'
        'ZChkdXJhdGlvbipzZWxmLmJhbmsucmF0ZSkKICAgICAgICAgICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsdG90YWwsc2VsZi5iYW5r'
        'LnJhdGUpOgogICAgICAgICAgICAgICAgZW5kPW1pbih0b3RhbCxzdGFydCtzZWxmLmJhbmsucmF0ZSkKICAgICAgICAgICAgICAg'
        'IGNodW5rPW5wLnplcm9zKGVuZC1zdGFydCxkdHlwZT0iPGkyIikKICAgICAgICAgICAgICAgIGZvciBzZWdtZW50IGluIHNlbGYu'
        'c2VnbWVudHM6CiAgICAgICAgICAgICAgICAgICAgYT1yb3VuZChzZWdtZW50WyJzdGFydCJdKnNlbGYuYmFuay5yYXRlKQogICAg'
        'ICAgICAgICAgICAgICAgIGI9bWluKHJvdW5kKHNlZ21lbnRbImVuZCJdKnNlbGYuYmFuay5yYXRlKSxhK2xlbihzZWdtZW50WyJz'
        'YW1wbGVzIl0pKQogICAgICAgICAgICAgICAgICAgIGxvLGhpPW1heChhLHN0YXJ0KSxtaW4oYixlbmQpCiAgICAgICAgICAgICAg'
        'ICAgICAgaWYgaGk+bG86IGNodW5rW2xvLXN0YXJ0OmhpLXN0YXJ0XT1zZWdtZW50WyJzYW1wbGVzIl1bbG8tYTpoaS1hXQogICAg'
        'ICAgICAgICAgICAgb3V0LndyaXRlZnJhbWVzKGNodW5rLnRvYnl0ZXMoKSkKCg=='
        , 'de874f5678fcb3d733e95a016cef7780e89750f7302617ec1e872f58df1217f7'
    ),
    'guide_core/types.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxk'
        'CmZyb20gZW51bSBpbXBvcnQgRW51bQppbXBvcnQgdGltZQppbXBvcnQgY3YyCmltcG9ydCBudW1weSBhcyBucAoKY2xhc3MgQWN0'
        'aW9uKHN0ciwgRW51bSk6CiAgICBVTkNMRUFSID0gIlVOQ0xFQVIiCiAgICBTVE9QID0gIlNUT1AiCiAgICBTTE9XID0gIlNMT1ci'
        'CiAgICBLRUVQX0hFQURJTkcgPSAiS0VFUF9IRUFESU5HIgogICAgU0hJRlRfTEVGVCA9ICJTSElGVF9MRUZUIgogICAgU0hJRlRf'
        'UklHSFQgPSAiU0hJRlRfUklHSFQiCgpTSURFX0dVSURBTkNFID0ge0FjdGlvbi5TSElGVF9MRUZULCBBY3Rpb24uU0hJRlRfUklH'
        'SFR9CkRJUkVDVElPTkFMID0ge0FjdGlvbi5LRUVQX0hFQURJTkcsICpTSURFX0dVSURBTkNFfQpTUEVFQ0hfUFJJT1JJVFkgPSB7'
        'CiAgICBBY3Rpb24uU1RPUDogMywKICAgIEFjdGlvbi5TSElGVF9MRUZUOiAzLAogICAgQWN0aW9uLlNISUZUX1JJR0hUOiAzLAp9'
        'Ck1FU1NBR0VTID0gewogICAgQWN0aW9uLlVOQ0xFQVI6ICJDaMawYSB4w6FjIG5o4bqtbiDEkcaw4bujYyBs4buRaSDEkWkuIiwK'
        'ICAgIEFjdGlvbi5TVE9QOiAiROG7q25nIGzhuqFpLiIsCiAgICBBY3Rpb24uU0xPVzogIsSQaSBjaOG6rW0gbOG6oWkuIiwKICAg'
        'IEFjdGlvbi5LRUVQX0hFQURJTkc6ICJHaeG7ryBoxrDhu5tuZywgxJFpIGNo4bqtbS4iLAogICAgQWN0aW9uLlNISUZUX0xFRlQ6'
        'ICJM4buHY2ggbmjhurkgc2FuZyB0csOhaS4iLAogICAgQWN0aW9uLlNISUZUX1JJR0hUOiAiTOG7h2NoIG5o4bq5IHNhbmcgcGjh'
        'uqNpLiIsCn0KCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIEZyYW1lOgogICAgc2Vzc2lvbjogc3RyCiAgICBzZXF1ZW5j'
        'ZTogaW50CiAgICBwdHM6IGZsb2F0CiAgICByZWNlaXZlZF9hdDogZmxvYXQKICAgIGJncjogbnAubmRhcnJheSA9IGZpZWxkKHJl'
        'cHI9RmFsc2UsIGNvbXBhcmU9RmFsc2UpCiAgICBvcmlnaW5hbF9odzogdHVwbGVbaW50LCBpbnRdID0gKDAsIDApCiAgICBtaXJy'
        'b3JlZF9pbnB1dDogYm9vbCA9IEZhbHNlCgogICAgQHByb3BlcnR5CiAgICBkZWYga2V5KHNlbGYpOgogICAgICAgIHJldHVybiAo'
        'c2VsZi5zZXNzaW9uLCBzZWxmLnNlcXVlbmNlKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGdlb21ldHJ5KHNlbGYpOgogICAgICAg'
        'IHJldHVybiAoc2VsZi5vcmlnaW5hbF9odywgc2VsZi5iZ3Iuc2hhcGVbOjJdLCBzZWxmLm1pcnJvcmVkX2lucHV0KQoKICAgIEBw'
        'cm9wZXJ0eQogICAgZGVmIHRyYW5zZm9ybShzZWxmKToKICAgICAgICBoLHc9c2VsZi5iZ3Iuc2hhcGVbOjJdCiAgICAgICAgb2gs'
        'b3c9c2VsZi5vcmlnaW5hbF9odyBpZiBzZWxmLm9yaWdpbmFsX2h3IT0oMCwwKSBlbHNlIChoLHcpCiAgICAgICAgc3gsc3k9dy9v'
        'dyxoL29oCiAgICAgICAgcmV0dXJuIHsib3JpZ2luYWxfaHciOltvaCxvd10sImFuYWx5c2lzX2h3IjpbaCx3XSwibWlycm9yX2Nv'
        'cnJlY3RlZCI6c2VsZi5taXJyb3JlZF9pbnB1dCwKICAgICAgICAgICAgICAgICJvcmlnaW5hbF90b19hbmFseXNpc19lZGdlcyI6'
        'W1stc3ggaWYgc2VsZi5taXJyb3JlZF9pbnB1dCBlbHNlIHN4LDAsdyBpZiBzZWxmLm1pcnJvcmVkX2lucHV0IGVsc2UgMF0sCiAg'
        'ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBbMCxzeSwwXSxbMCwwLDFdXX0KCiAgICBAY2xhc3Nt'
        'ZXRob2QKICAgIGRlZiBwcmVwYXJlKGNscywgaW1hZ2UsIHNlc3Npb24sIHNlcXVlbmNlLCBwdHMsIG1heF9zaWRlPTY0MCwgbWly'
        'cm9yPUZhbHNlLCByZWNlaXZlZF9hdD1Ob25lKToKICAgICAgICBvcmlnaW5hbCA9IGltYWdlLnNoYXBlWzoyXQogICAgICAgIGlm'
        'IG1pcnJvcjoKICAgICAgICAgICAgaW1hZ2UgPSBjdjIuZmxpcChpbWFnZSwgMSkgICMgQ2Fub25pY2FsIGNhbWVyYS1yZWxhdGl2'
        'ZSBsZWZ0L3JpZ2h0IGJlZm9yZSBhbGwgbW9kZWxzLgogICAgICAgIHNjYWxlID0gbWluKDEuLCBtYXhfc2lkZSAvIG1heChvcmln'
        'aW5hbCkpCiAgICAgICAgaWYgc2NhbGUgPCAxOgogICAgICAgICAgICBpbWFnZSA9IGN2Mi5yZXNpemUoaW1hZ2UsIChyb3VuZChv'
        'cmlnaW5hbFsxXSpzY2FsZSksIHJvdW5kKG9yaWdpbmFsWzBdKnNjYWxlKSkpCiAgICAgICAgcmV0dXJuIGNscyhzZXNzaW9uLCBz'
        'ZXF1ZW5jZSwgcHRzLCB0aW1lLnBlcmZfY291bnRlcigpIGlmIHJlY2VpdmVkX2F0IGlzIE5vbmUgZWxzZSByZWNlaXZlZF9hdCwK'
        'ICAgICAgICAgICAgICAgICAgIGltYWdlLCBvcmlnaW5hbCwgbWlycm9yKQoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3Mg'
        'RGV0ZWN0aW9uOgogICAgbGFiZWw6IHN0cgogICAgY29uZmlkZW5jZTogZmxvYXQKICAgIGJveDogdHVwbGVbZmxvYXQsIGZsb2F0'
        'LCBmbG9hdCwgZmxvYXRdCiAgICBzb3VyY2VzOiB0dXBsZVtzdHIsIC4uLl0KICAgIHRyYWNrX2lkOiBpbnQgfCBOb25lID0gTm9u'
        'ZQogICAgdmVsb2NpdHk6IHR1cGxlW2Zsb2F0LCBmbG9hdF0gPSAoMC4sIDAuKQoKQGRhdGFjbGFzcwpjbGFzcyBNb2RlbFJlc3Vs'
        'dDoKICAgIG1vZGVsX2lkOiBzdHIKICAgIHJvbGU6IHN0cgogICAgZnJhbWVfa2V5OiB0dXBsZVtzdHIsIGludF0KICAgIGdlb21l'
        'dHJ5OiB0dXBsZQogICAgc3RhcnRlZF9hdDogZmxvYXQKICAgIGZpbmlzaGVkX2F0OiBmbG9hdAogICAgcGF5bG9hZDogb2JqZWN0'
        'ID0gTm9uZQogICAgZXJyb3I6IHN0ciB8IE5vbmUgPSBOb25lCiAgICBza2lwcGVkOiBzdHIgfCBOb25lID0gTm9uZQoKQGRhdGFj'
        'bGFzcwpjbGFzcyBTY2VuZToKICAgIGZyYW1lOiBGcmFtZQogICAgZGV0ZWN0aW9uczogbGlzdFtEZXRlY3Rpb25dCiAgICBzZW1h'
        'bnRpYzogZGljdCB8IE5vbmUgPSBOb25lCiAgICBkZXB0aDogZGljdCB8IE5vbmUgPSBOb25lCiAgICBlcnJvcnM6IGxpc3Rbc3Ry'
        'XSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgbW90aW9uX2ludmFsaWQ6IGJvb2wgPSBGYWxzZQogICAgYmx1cl9z'
        'Y29yZTogZmxvYXQgPSAxMDAuCiAgICBicmlnaHRuZXNzOiBmbG9hdCA9IDEwMC4KICAgIGRldGVjdG9yX3ZhbGlkOiBib29sID0g'
        'VHJ1ZQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIG9ic2VydmF0aW9uX3ZhbGlkKHNlbGYpOgogICAgICAgIHJldHVybiBzZWxmLmRl'
        'dGVjdG9yX3ZhbGlkIGFuZCBub3Qgc2VsZi5tb3Rpb25faW52YWxpZCBhbmQgc2VsZi5ibHVyX3Njb3JlID49IDI1IGFuZCBzZWxm'
        'LmJyaWdodG5lc3MgPj0gMTgKCkBkYXRhY2xhc3MKY2xhc3MgUm91dGU6CiAgICBhY3Rpb246IEFjdGlvbiB8IE5vbmUKICAgIHBh'
        'dGg6IGxpc3RbdHVwbGVbaW50LCBpbnRdXSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgbWFzazogbnAubmRhcnJh'
        'eSB8IE5vbmUgPSBmaWVsZChkZWZhdWx0PU5vbmUsIHJlcHI9RmFsc2UpCiAgICBzdGF0ZXM6IG5wLm5kYXJyYXkgfCBOb25lID0g'
        'ZmllbGQoZGVmYXVsdD1Ob25lLCByZXByPUZhbHNlKQogICAgY2xlYXJhbmNlOiBmbG9hdCA9IDAuCiAgICBuYXJyb3c6IGJvb2wg'
        'PSBGYWxzZQogICAgcmVhc29uOiBzdHIgPSAiIgogICAgZXZpZGVuY2VfY29tcGxldGU6IGJvb2wgPSBGYWxzZQoKQGRhdGFjbGFz'
        'cyhmcm96ZW49VHJ1ZSkKY2xhc3MgRGVjaXNpb246CiAgICBhY3Rpb246IEFjdGlvbgogICAgY2xvY2s6IGZsb2F0CiAgICB2YWxp'
        'ZF91bnRpbDogZmxvYXQKICAgIGZyYW1lX2tleTogdHVwbGVbc3RyLCBpbnRdCiAgICByZWFzb246IHN0cgogICAgcm91dGU6IFJv'
        'dXRlIHwgTm9uZSA9IGZpZWxkKGRlZmF1bHQ9Tm9uZSwgY29tcGFyZT1GYWxzZSkKICAgIHByaW9yaXR5OiBpbnQgPSAxCiAgICBp'
        'bnZhbGlkYXRlc19yb3V0ZTogYm9vbCA9IEZhbHNlCgogICAgQHByb3BlcnR5CiAgICBkZWYgdGV4dChzZWxmKToKICAgICAgICBy'
        'ZXR1cm4gTUVTU0FHRVNbc2VsZi5hY3Rpb25dCgogICAgZGVmIHJlY29yZChzZWxmKToKICAgICAgICByZXR1cm4geyJhY3Rpb24i'
        'OiBzZWxmLmFjdGlvbi52YWx1ZSwgInRpbWVfcyI6IHNlbGYuY2xvY2ssICJ2YWxpZF91bnRpbCI6IHNlbGYudmFsaWRfdW50aWws'
        'CiAgICAgICAgICAgICAgICAiZnJhbWVfa2V5IjogbGlzdChzZWxmLmZyYW1lX2tleSksICJyZWFzb24iOiBzZWxmLnJlYXNvbiwg'
        'InByaW9yaXR5Ijogc2VsZi5wcmlvcml0eSwKICAgICAgICAgICAgICAgICJpbnZhbGlkYXRlc19yb3V0ZSI6IHNlbGYuaW52YWxp'
        'ZGF0ZXNfcm91dGUsCiAgICAgICAgICAgICAgICAicGF0aCI6IHNlbGYucm91dGUucGF0aCBpZiBzZWxmLnJvdXRlIGVsc2UgW119'
        'Cgo='
        , '7628ef43448f2ea81c44b3258846792a2452f1ce6b2a4949b0b8a182841c5555'
    ),
    'guide_core/policy.py': (
        'ZnJvbSAudHlwZXMgaW1wb3J0IEFjdGlvbiwgRGVjaXNpb24sIERJUkVDVElPTkFMLCBTUEVFQ0hfUFJJT1JJVFkKZnJvbSAucGxh'
        'bm5pbmcgaW1wb3J0IHVyZ2VudCwgcm91dGVfYmxvY2tlZAoKY2xhc3MgRGVjaXNpb25Qb2xpY3k6CiAgICBkZWYgX19pbml0X18o'
        'c2VsZixjb25maWcpOgogICAgICAgIHNlbGYuYz1jb25maWcKICAgICAgICBzZWxmLmFjdGl2ZT1Ob25lCiAgICAgICAgc2VsZi5j'
        'YW5kaWRhdGU9Tm9uZQogICAgICAgIHNlbGYuc2luY2U9MC4KICAgICAgICBzZWxmLmNvdW50PTAKICAgICAgICBzZWxmLmxhc3Rf'
        'a2V5PU5vbmUKICAgICAgICBzZWxmLmNhbmRpZGF0ZV9yb3V0ZT1Ob25lCiAgICAgICAgc2VsZi5sYXN0X3B0cz1Ob25lCgogICAg'
        'QHN0YXRpY21ldGhvZAogICAgZGVmIGNvbnNpc3RlbnQoYSxiLHdpZHRoKToKICAgICAgICBpZiBub3QgYSBvciBub3QgYiBvciBu'
        'b3QgYS5wYXRoIG9yIG5vdCBiLnBhdGg6IHJldHVybiBUcnVlCiAgICAgICAgaW1wb3J0IG51bXB5IGFzIG5wCiAgICAgICAgeD1u'
        'cC5hcnJheShhLnBhdGgpWzosMF07eT1ucC5hcnJheShiLnBhdGgpWzosMF0KICAgICAgICByZXR1cm4gbGVuKHgpPT1sZW4oeSkg'
        'YW5kIGZsb2F0KG5wLm1heChucC5hYnMoeC15KSkpPD13aWR0aCouMDYKCiAgICBkZWYgX2VtaXQoc2VsZixhY3Rpb24sZnJhbWUs'
        'Y2xvY2sscmVhc29uLHJvdXRlPU5vbmUpOgogICAgICAgIHByaW9yaXR5PVNQRUVDSF9QUklPUklUWS5nZXQoYWN0aW9uLDEpCiAg'
        'ICAgICAgIyBQbGFubmVyIFNUT1AgbWVhbnMgZm9yd2FyZCBtb3Rpb24gaXMgdW5zYWZlIG9yIGV4aXN0aW5nIHJvdXRlIGV2aWRl'
        'bmNlIHdhcyBsb3N0LgogICAgICAgIGRlY2lzaW9uPURlY2lzaW9uKGFjdGlvbixjbG9jayxjbG9jaytzZWxmLmMubWF4X2FnZV9z'
        'LGZyYW1lLmtleSxyZWFzb24scm91dGUscHJpb3JpdHksCiAgICAgICAgICAgICAgICAgICAgICAgICAgaW52YWxpZGF0ZXNfcm91'
        'dGU9YWN0aW9uID09IEFjdGlvbi5TVE9QKQogICAgICAgIHNlbGYuYWN0aXZlPWRlY2lzaW9uCiAgICAgICAgaWYgYWN0aW9uIG5v'
        'dCBpbiBESVJFQ1RJT05BTCBhbmQgcmVhc29uIT0iY29uZmlybWluZ19jb3JyaWRvciI6CiAgICAgICAgICAgIHNlbGYucmVzZXRf'
        'Y29uZmlybWF0aW9uKCkKICAgICAgICByZXR1cm4gZGVjaXNpb24KCiAgICBkZWYgcmVzZXRfY29uZmlybWF0aW9uKHNlbGYpOgog'
        'ICAgICAgIHNlbGYuY2FuZGlkYXRlID0gTm9uZQogICAgICAgIHNlbGYuY291bnQgPSAwCgogICAgZGVmIHN0YWxlKHNlbGYsZnJh'
        'bWUsY2xvY2sscmVhc29uPSJvYnNlcnZhdGlvbl9leHBpcmVkIik6CiAgICAgICAgYWN0aW9uPUFjdGlvbi5TVE9QIGlmIHNlbGYu'
        'YWN0aXZlIGFuZCBzZWxmLmFjdGl2ZS5hY3Rpb24gaW4gRElSRUNUSU9OQUwgZWxzZSBBY3Rpb24uVU5DTEVBUgogICAgICAgIHJl'
        'dHVybiBzZWxmLl9lbWl0KGFjdGlvbixmcmFtZSxjbG9jayxyZWFzb24pCgogICAgZGVmIHVwZGF0ZShzZWxmLHNjZW5lLHJvdXRl'
        'LGNsb2NrLGZ1bGw9VHJ1ZSk6CiAgICAgICAgZnJhbWU9c2NlbmUuZnJhbWUKICAgICAgICBwcmV2aW91cz1zZWxmLmFjdGl2ZQog'
        'ICAgICAgIGlmIHVyZ2VudChzY2VuZSk6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9lbWl0KEFjdGlvbi5TVE9QLGZyYW1lLGNs'
        'b2NrLCJuZWFyX2NlbnRlcl9vYnN0YWNsZSIpCiAgICAgICAgaWYgbm90IHNjZW5lLm9ic2VydmF0aW9uX3ZhbGlkOgogICAgICAg'
        'ICAgICByZXR1cm4gc2VsZi5zdGFsZShmcmFtZSxjbG9jaywicG9vcl9vcl9taXNzaW5nX29ic2VydmF0aW9uIikKICAgICAgICBp'
        'ZiBwcmV2aW91cyBhbmQgcHJldmlvdXMuYWN0aW9uIGluIERJUkVDVElPTkFMIGFuZCByb3V0ZV9ibG9ja2VkKHByZXZpb3VzLnJv'
        'dXRlLHNjZW5lKToKICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2VtaXQoQWN0aW9uLlNUT1AsZnJhbWUsY2xvY2ssInByZXZpb3Vz'
        'X3JvdXRlX2Jsb2NrZWQiKQogICAgICAgIGlmIG5vdCBmdWxsOgogICAgICAgICAgICAjIEZhc3QgZGV0ZWN0b3IgY2FuIHZldG8g'
        'YnV0IGNhbiBuZXZlciByZWZyZXNoIHRoZSBkaXJlY3Rpb25hbCBldmlkZW5jZSBleHBpcnkuCiAgICAgICAgICAgIGlmIHByZXZp'
        'b3VzIGFuZCBwcmV2aW91cy5hY3Rpb24gaW4gRElSRUNUSU9OQUw6CiAgICAgICAgICAgICAgICBpZiBjbG9jaz5wcmV2aW91cy52'
        'YWxpZF91bnRpbDogcmV0dXJuIHNlbGYuc3RhbGUoZnJhbWUsY2xvY2spCiAgICAgICAgICAgICAgICByZXR1cm4gcHJldmlvdXMK'
        'ICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2VtaXQoQWN0aW9uLlNMT1cgaWYgc2NlbmUuZGV0ZWN0aW9ucyBlbHNlIEFjdGlvbi5V'
        'TkNMRUFSLGZyYW1lLGNsb2NrLCJ3YWl0aW5nX2NvbnRleHQiKQogICAgICAgIGlmIHJvdXRlIGlzIE5vbmUgb3Igcm91dGUuYWN0'
        'aW9uIGlzIE5vbmUgb3Igcm91dGUubmFycm93OgogICAgICAgICAgICBpZiBwcmV2aW91cyBhbmQgcHJldmlvdXMuYWN0aW9uIGlu'
        'IERJUkVDVElPTkFMOgogICAgICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2VtaXQoQWN0aW9uLlNUT1AsZnJhbWUsY2xvY2ssInJv'
        'dXRlX25vX2xvbmdlcl9jb25maXJtZWQiKQogICAgICAgICAgICBhY3Rpb249QWN0aW9uLlNMT1cgaWYgcm91dGUgYW5kIHJvdXRl'
        'LmV2aWRlbmNlX2NvbXBsZXRlIGVsc2UgQWN0aW9uLlVOQ0xFQVIKICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2VtaXQoYWN0aW9u'
        'LGZyYW1lLGNsb2NrLHJvdXRlLnJlYXNvbiBpZiByb3V0ZSBlbHNlICJjb250ZXh0X3VuYXZhaWxhYmxlIixyb3V0ZSkKICAgICAg'
        'ICBpZiBwcmV2aW91cyBhbmQgcHJldmlvdXMuYWN0aW9uIGluIERJUkVDVElPTkFMIGFuZCBwcmV2aW91cy5hY3Rpb24hPXJvdXRl'
        'LmFjdGlvbjoKICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2VtaXQoQWN0aW9uLlNUT1AsZnJhbWUsY2xvY2ssImRpcmVjdGlvbl9j'
        'aGFuZ2VkX3JlcXVpcmVzX2NvbmZpcm1hdGlvbiIpCiAgICAgICAgaWYgcHJldmlvdXMgYW5kIHByZXZpb3VzLmFjdGlvbiBpbiBE'
        'SVJFQ1RJT05BTCBhbmQgbm90IHNlbGYuY29uc2lzdGVudChwcmV2aW91cy5yb3V0ZSxyb3V0ZSxmcmFtZS5iZ3Iuc2hhcGVbMV0p'
        'OgogICAgICAgICAgICByZXR1cm4gc2VsZi5fZW1pdChBY3Rpb24uU1RPUCxmcmFtZSxjbG9jaywiY29ycmlkb3JfY2hhbmdlZF9y'
        'ZXF1aXJlc19jb25maXJtYXRpb24iKQogICAgICAgIGdhcD1zZWxmLmxhc3RfcHRzIGlzIG5vdCBOb25lIGFuZCBub3QgMDw9ZnJh'
        'bWUucHRzLXNlbGYubGFzdF9wdHM8PXNlbGYuYy50cmFja190dGxfcwogICAgICAgIGlmIHNlbGYuY2FuZGlkYXRlIT1yb3V0ZS5h'
        'Y3Rpb24gb3IgZ2FwIG9yIG5vdCBzZWxmLmNvbnNpc3RlbnQoc2VsZi5jYW5kaWRhdGVfcm91dGUscm91dGUsZnJhbWUuYmdyLnNo'
        'YXBlWzFdKToKICAgICAgICAgICAgc2VsZi5jYW5kaWRhdGU9cm91dGUuYWN0aW9uCiAgICAgICAgICAgIHNlbGYuc2luY2U9ZnJh'
        'bWUucHRzCiAgICAgICAgICAgIHNlbGYuY291bnQ9MAogICAgICAgIHNlbGYuY2FuZGlkYXRlX3JvdXRlPXJvdXRlCiAgICAgICAg'
        'c2VsZi5sYXN0X3B0cz1mcmFtZS5wdHMKICAgICAgICBpZiBmcmFtZS5rZXkhPXNlbGYubGFzdF9rZXk6CiAgICAgICAgICAgIHNl'
        'bGYuY291bnQrPTEKICAgICAgICAgICAgc2VsZi5sYXN0X2tleT1mcmFtZS5rZXkKICAgICAgICBpZiBzZWxmLmNvdW50Pj1zZWxm'
        'LmMuc3RhYmxlX29ic2VydmF0aW9ucyBhbmQgZnJhbWUucHRzLXNlbGYuc2luY2U+PXNlbGYuYy5zdGFibGVfcy0xZS04OgogICAg'
        'ICAgICAgICByZXR1cm4gc2VsZi5fZW1pdChyb3V0ZS5hY3Rpb24sZnJhbWUsY2xvY2sscm91dGUucmVhc29uLHJvdXRlKQogICAg'
        'ICAgIHJldHVybiBzZWxmLl9lbWl0KEFjdGlvbi5TTE9XLGZyYW1lLGNsb2NrLCJjb25maXJtaW5nX2NvcnJpZG9yIixyb3V0ZSkK'
        'Cg=='
        , 'd78e72dd01b4761947bb3e8a622147ea8307c7251c69b20bd208180c28c479eb'
    ),
    'guide_core/demo.py': (
        'IiIicHl0aG9uIC1tIGd1aWRlX2NvcmUuZGVtbyAtLWNvbmZpZyBjb25maWdzL2d1aWRlX2thZ2dsZS5qc29uIC0tdmlkZW8gcGF0'
        'aC5tcDQgLS1vdXRwdXQgcnVucy9ndWlkZV92MiIiIgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSBkYXRhY2xhc3NlcyBp'
        'bXBvcnQgcmVwbGFjZQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgaGFzaGxp'
        'YgppbXBvcnQgaW1wb3J0bGliLm1ldGFkYXRhCmltcG9ydCBqc29uCmltcG9ydCBtYXRoCmltcG9ydCBzdWJwcm9jZXNzCmltcG9y'
        'dCB0aW1lCmltcG9ydCB1dWlkCmltcG9ydCBjdjIKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gLmNvbmZpZyBpbXBvcnQgQ29uZmln'
        'CmZyb20gLnR5cGVzIGltcG9ydCBGcmFtZSwgRElSRUNUSU9OQUwKZnJvbSAuZW5naW5lIGltcG9ydCBHdWlkZUVuZ2luZQpmcm9t'
        'IC5hdWRpbyBpbXBvcnQgQXVkaW9UaW1lbGluZSwgQ2xpcEJhbmsKZnJvbSAucmVuZGVyIGltcG9ydCByZW5kZXIKZnJvbSAucmVw'
        'bGF5IGltcG9ydCBydW5fcmVwbGF5CgpkZWYgZGlzdHJpYnV0aW9uKHZhbHVlcyk6CiAgICByZXR1cm4geyJjb3VudCI6bGVuKHZh'
        'bHVlcyksInA1MCI6ZmxvYXQobnAucGVyY2VudGlsZSh2YWx1ZXMsNTApKSBpZiB2YWx1ZXMgZWxzZSBOb25lLAogICAgICAgICAg'
        'ICAicDk1IjpmbG9hdChucC5wZXJjZW50aWxlKHZhbHVlcyw5NSkpIGlmIHZhbHVlcyBlbHNlIE5vbmV9CgpkZWYgZmluZ2VycHJp'
        'bnQocGF0aCk6CiAgICBwYXRoPVBhdGgocGF0aCkKICAgIGZpbGVzPVtwYXRoXSBpZiBwYXRoLmlzX2ZpbGUoKSBlbHNlIHNvcnRl'
        'ZChwYXRoLmdsb2IoIiouc2FmZXRlbnNvcnMiKSkKICAgIGRpZ2VzdHM9e30KICAgIGZvciBpdGVtIGluIGZpbGVzOgogICAgICAg'
        'IGRpZ2VzdD1oYXNobGliLnNoYTI1NigpCiAgICAgICAgd2l0aCBpdGVtLm9wZW4oInJiIikgYXMgZjoKICAgICAgICAgICAgZm9y'
        'IGNodW5rIGluIGl0ZXIobGFtYmRhOmYucmVhZCgxMDI0KjEwMjQpLGIiIik6IGRpZ2VzdC51cGRhdGUoY2h1bmspCiAgICAgICAg'
        'ZGlnZXN0c1tpdGVtLm5hbWVdPWRpZ2VzdC5oZXhkaWdlc3QoKQogICAgcmV0dXJuIGRpZ2VzdHMKCmRlZiBydW4oY29uZmlnLHZp'
        'ZGVvLG91dHB1dCxzZWNvbmRzPU5vbmUscHJvZ3Jlc3M9VHJ1ZSk6CiAgICBvdXRwdXQ9UGF0aChvdXRwdXQpCiAgICBvdXRwdXQu'
        'bWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICBpZiAob3V0cHV0LyJkZWNpc2lvbnMuanNvbmwiKS5leGlzdHMo'
        'KToKICAgICAgICByYWlzZSBGaWxlRXhpc3RzRXJyb3IoIkNob29zZSBhIGZyZXNoIG91dHB1dCBkaXJlY3RvcnkgdG8gcHJlc2Vy'
        'dmUgcHJldmlvdXMgbWVhc3VyZW1lbnRzOiAiK3N0cihvdXRwdXQpKQogICAgdmlkZW89UGF0aCh2aWRlbykKICAgIGlmIG5vdCB2'
        'aWRlby5pc19maWxlKCk6IHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKHZpZGVvKQogICAgY29uZmlnLnZhbGlkYXRlKCkKICAgIGlt'
        'cG9ydCB0b3JjaAogICAgaW1wb3J0IHBzdXRpbAogICAgdG9yY2guc2V0X251bV90aHJlYWRzKDQpCiAgICBjdjIuc2V0TnVtVGhy'
        'ZWFkcygxKQogICAgcHJvY2Vzcz1wc3V0aWwuUHJvY2VzcygpCiAgICByc3M9W10KICAgIGRlZiBtZW1vcnkoKToKICAgICAgICBy'
        'c3MuYXBwZW5kKHByb2Nlc3MubWVtb3J5X2luZm8oKS5yc3MvMTAyNCoqMikKICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxl'
        'KCk6IHRvcmNoLmN1ZGEucmVzZXRfcGVha19tZW1vcnlfc3RhdHMoKQogICAgZW5naW5lPUd1aWRlRW5naW5lKGNvbmZpZykKICAg'
        'IGxvYWRfc3RhcnQ9dGltZS5wZXJmX2NvdW50ZXIoKQogICAgdHJ5OgogICAgICAgIGVuZ2luZS5sb2FkKCkKICAgIGV4Y2VwdCBF'
        'eGNlcHRpb246CiAgICAgICAgZW5naW5lLmNsb3NlKCkKICAgICAgICByYWlzZQogICAgbG9hZF9zZWNvbmRzPXRpbWUucGVyZl9j'
        'b3VudGVyKCktbG9hZF9zdGFydAogICAgYmFuaz1DbGlwQmFuayhjb25maWcuYXVkaW9fZGlyKSBpZiBjb25maWcuYXVkaW9fZW5h'
        'YmxlZCBlbHNlIE5vbmUKICAgIGF1ZGlvPUF1ZGlvVGltZWxpbmUoYmFuaykgaWYgYmFuayBlbHNlIE5vbmUKICAgIChvdXRwdXQv'
        'ImNvbmZpZy5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKGNvbmZpZy5hc19kaWN0KCksZW5zdXJlX2FzY2lpPUZhbHNlLGlu'
        'ZGVudD0yKSxlbmNvZGluZz0idXRmLTgiKQogICAgcmVjb3Jkcz1bXQogICAgcGlwZWxpbmVfbXM9W10KICAgIHN0YWxlX2NvdW50'
        'PTAKICAgIGZyYW1lc191c2VkPVtdCiAgICByZWNlaXZlZF9sYXRlbmN5PVtdCiAgICBjb3VudHM9Q291bnRlcigpCiAgICB3cml0'
        'ZXI9Tm9uZQogICAgdmlkZW9fZnJhbWVzPTAKICAgIHByZXZpb3VzX2ltYWdlPU5vbmUKICAgIHdyaXRlcl9mcHM9Y29uZmlnLnNh'
        'bXBsZV9mcHMKICAgIG9yaWdpbj0wLgogICAgZWxhcHNlZF9zdGFydD10aW1lLnBlcmZfY291bnRlcigpCiAgICBzdGF0cz17fQog'
        'ICAgZGVmIHdyaXRlX3VudGlsKGltYWdlLHQpOgogICAgICAgIG5vbmxvY2FsIHdyaXRlcix2aWRlb19mcmFtZXMscHJldmlvdXNf'
        'aW1hZ2UKICAgICAgICBpZiB3cml0ZXIgaXMgTm9uZToKICAgICAgICAgICAgd3JpdGVyPWN2Mi5WaWRlb1dyaXRlcihzdHIob3V0'
        'cHV0LyJwcmV2aWV3X3Jhdy5tcDQiKSxjdjIuVmlkZW9Xcml0ZXJfZm91cmNjKCoibXA0diIpLHdyaXRlcl9mcHMsKGltYWdlLnNo'
        'YXBlWzFdLGltYWdlLnNoYXBlWzBdKSkKICAgICAgICAgICAgaWYgbm90IHdyaXRlci5pc09wZW5lZCgpOiByYWlzZSBSdW50aW1l'
        'RXJyb3IoIlZpZGVvIGVuY29kZXIgdW5hdmFpbGFibGUiKQogICAgICAgIHRhcmdldD1tYXRoLmZsb29yKHQqd3JpdGVyX2Zwcysx'
        'ZS02KQogICAgICAgIHdoaWxlIHZpZGVvX2ZyYW1lczx0YXJnZXQ6CiAgICAgICAgICAgIHdyaXRlci53cml0ZShwcmV2aW91c19p'
        'bWFnZSBpZiBwcmV2aW91c19pbWFnZSBpcyBub3QgTm9uZSBlbHNlIGltYWdlKQogICAgICAgICAgICB2aWRlb19mcmFtZXMrPTEK'
        'ICAgICAgICBwcmV2aW91c19pbWFnZT1pbWFnZQoKICAgIGRlZiBlbWl0KHNjZW5lLGRlY2lzaW9uLHJlc3VsdHMsYXQsc3RhZ2Up'
        'OgogICAgICAgIG5vbmxvY2FsIHN0YWxlX2NvdW50CiAgICAgICAgbWVtb3J5KCkKICAgICAgICBpZiBjb25maWcubW9kZT09InJl'
        'YWx0aW1lX3JlcGxheSI6CiAgICAgICAgICAgIG5vdz10aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgIGF0PW5vdy1vcmln'
        'aW4KICAgICAgICAgICAgaWYgbm93PmRlY2lzaW9uLnZhbGlkX3VudGlsOgogICAgICAgICAgICAgICAgZGVjaXNpb249ZW5naW5l'
        'LnBvbGljeS5zdGFsZShzY2VuZS5mcmFtZSxub3csImV4cGlyZWRfYmVmb3JlX291dHB1dCIpCiAgICAgICAgICAgICAgICBzdGFn'
        'ZT0ic3RhbGVfYmVmb3JlX291dHB1dCIKICAgICAgICAgICAgZGVjaXNpb249cmVwbGFjZShkZWNpc2lvbixjbG9jaz1kZWNpc2lv'
        'bi5jbG9jay1vcmlnaW4sdmFsaWRfdW50aWw9ZGVjaXNpb24udmFsaWRfdW50aWwtb3JpZ2luKQogICAgICAgICAgICBsYXRlbmN5'
        'PShub3ctc2NlbmUuZnJhbWUucmVjZWl2ZWRfYXQpKjEwMDAKICAgICAgICAgICAgcmVjZWl2ZWRfbGF0ZW5jeS5hcHBlbmQobGF0'
        'ZW5jeSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBsYXRlbmN5PU5vbmUKICAgICAgICBpZiAic3RhbGUiIGluIHN0YWdlOiBz'
        'dGFsZV9jb3VudCs9MQogICAgICAgIGlmIGF1ZGlvOiBhdWRpby51cGRhdGUoZGVjaXNpb24sYXQpCiAgICAgICAgcmVjb3JkPWRl'
        'Y2lzaW9uLnJlY29yZCgpCiAgICAgICAgcmVjb3JkLnVwZGF0ZShlbWl0dGVkX2F0X3M9YXQsdmlkZW9fdGltZV9zPXNjZW5lLmZy'
        'YW1lLnB0cyxzdGFnZT1zdGFnZSwKICAgICAgICAgICAgICAgICAgICAgIGdlb21ldHJ5PXNjZW5lLmZyYW1lLnRyYW5zZm9ybSxy'
        'ZWNlaXZlZF9hdF9tb25vdG9uaWM9c2NlbmUuZnJhbWUucmVjZWl2ZWRfYXQsCiAgICAgICAgICAgICAgICAgICAgICBmcmFtZV90'
        'b19kZWNpc2lvbl9tcz1sYXRlbmN5LGVycm9ycz1zY2VuZS5lcnJvcnMsCiAgICAgICAgICAgICAgICAgICAgICBtb2RlbHM9W3si'
        'aWQiOnIubW9kZWxfaWQsInJvbGUiOnIucm9sZSwiZnJhbWVfa2V5IjpsaXN0KHIuZnJhbWVfa2V5KSwKICAgICAgICAgICAgICAg'
        'ICAgICAgICAgICAgICAgICJzdGFydGVkX2F0X21vbm90b25pYyI6ci5zdGFydGVkX2F0LCJmaW5pc2hlZF9hdF9tb25vdG9uaWMi'
        'OnIuZmluaXNoZWRfYXQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAibXMiOihyLmZpbmlzaGVkX2F0LXIuc3RhcnRl'
        'ZF9hdCkqMTAwMCwiZXJyb3IiOnIuZXJyb3IsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAic2tpcHBlZCI6ci5za2lw'
        'cGVkfSBmb3IgciBpbiByZXN1bHRzXSwKICAgICAgICAgICAgICAgICAgICAgIGRldGVjdGlvbnM9W3sibGFiZWwiOmQubGFiZWws'
        'ImNvbmZpZGVuY2UiOmQuY29uZmlkZW5jZSwiYm94IjpkLmJveCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAi'
        'dHJhY2tfaWQiOmQudHJhY2tfaWQsInNvdXJjZXMiOmQuc291cmNlcywidmVsb2NpdHkiOmQudmVsb2NpdHl9IGZvciBkIGluIHNj'
        'ZW5lLmRldGVjdGlvbnNdKQogICAgICAgIHJlY29yZHMuYXBwZW5kKHJlY29yZCkKICAgICAgICBjb3VudHNbZGVjaXNpb24uYWN0'
        'aW9uLnZhbHVlXSs9MQogICAgICAgIGZyYW1lc191c2VkLmFwcGVuZChzY2VuZS5mcmFtZS5zZXF1ZW5jZSkKICAgICAgICB3cml0'
        'ZV91bnRpbChyZW5kZXIoc2NlbmUsZGVjaXNpb24sc3RhZ2UpLGF0KQogICAgICAgIGlmIHByb2dyZXNzIGFuZCBsZW4ocmVjb3Jk'
        'cyklMTA9PTA6CiAgICAgICAgICAgIHByaW50KGYiUHJvY2Vzc2VkIHtsZW4ocmVjb3Jkcyl9IG91dHB1dHMgfCB2aWRlbyB7c2Nl'
        'bmUuZnJhbWUucHRzOi4xZn1zIHwge2RlY2lzaW9uLmFjdGlvbi52YWx1ZX0iLGZsdXNoPVRydWUpCiAgICB0cnk6CiAgICAgICAg'
        'aWYgY29uZmlnLm1vZGU9PSJvZmZsaW5lIjoKICAgICAgICAgICAgY2FwPWN2Mi5WaWRlb0NhcHR1cmUoc3RyKHZpZGVvKSkKICAg'
        'ICAgICAgICAgaWYgbm90IGNhcC5pc09wZW5lZCgpOiByYWlzZSBSdW50aW1lRXJyb3IoIkNhbm5vdCBvcGVuIHZpZGVvIikKICAg'
        'ICAgICAgICAgZnBzPWNhcC5nZXQoY3YyLkNBUF9QUk9QX0ZQUykgb3IgMzAuCiAgICAgICAgICAgIHN0cmlkZT1tYXgoMSxtYXRo'
        'LmNlaWwoZnBzL2NvbmZpZy5zYW1wbGVfZnBzKSkKICAgICAgICAgICAgd3JpdGVyX2Zwcz1mcHMvc3RyaWRlCiAgICAgICAgICAg'
        'IHNlcT0wCiAgICAgICAgICAgIGRlY29kZV9zZWNvbmRzPTAuCiAgICAgICAgICAgIGRlY29kZWQ9MAogICAgICAgICAgICBsYXN0'
        'X3B0cz0wLgogICAgICAgICAgICBzZXNzaW9uPSJvZmZsaW5lLSIrdXVpZC51dWlkNCgpLmhleFs6MTJdCiAgICAgICAgICAgIHRy'
        'eToKICAgICAgICAgICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgICAgICAgICAgcHRzPXNlcS9mcHMKICAgICAgICAgICAg'
        'ICAgICAgICBpZiBzZWNvbmRzIGlzIG5vdCBOb25lIGFuZCBwdHM+PXNlY29uZHM6IGJyZWFrCiAgICAgICAgICAgICAgICAgICAg'
        'dD10aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgICAgICAgICAgb2ssaW1hZ2U9Y2FwLnJlYWQoKQogICAgICAgICAgICAg'
        'ICAgICAgIGRlY29kZV9zZWNvbmRzKz10aW1lLnBlcmZfY291bnRlcigpLXQKICAgICAgICAgICAgICAgICAgICBpZiBub3Qgb2s6'
        'IGJyZWFrCiAgICAgICAgICAgICAgICAgICAgZGVjb2RlZCs9MQogICAgICAgICAgICAgICAgICAgIGlmIHNlcSVzdHJpZGU9PTA6'
        'CiAgICAgICAgICAgICAgICAgICAgICAgIGZyYW1lPUZyYW1lLnByZXBhcmUoaW1hZ2Usc2Vzc2lvbixzZXEscHRzLGNvbmZpZy5t'
        'YXhfc2lkZSxjb25maWcubWlycm9yX2lucHV0KQogICAgICAgICAgICAgICAgICAgICAgICB0PXRpbWUucGVyZl9jb3VudGVyKCkK'
        'ICAgICAgICAgICAgICAgICAgICAgICAgc2NlbmUsZGVjaXNpb24scmVzdWx0cz1lbmdpbmUucHJvY2VzcyhmcmFtZSkKICAgICAg'
        'ICAgICAgICAgICAgICAgICAgcGlwZWxpbmVfbXMuYXBwZW5kKCh0aW1lLnBlcmZfY291bnRlcigpLXQpKjEwMDApCiAgICAgICAg'
        'ICAgICAgICAgICAgICAgIGVtaXQoc2NlbmUsZGVjaXNpb24scmVzdWx0cyxwdHMsIm9mZmxpbmVfZnVzZWQiKQogICAgICAgICAg'
        'ICAgICAgICAgICAgICBsYXN0X3B0cz1wdHMKICAgICAgICAgICAgICAgICAgICBzZXErPTEKICAgICAgICAgICAgZmluYWxseToK'
        'ICAgICAgICAgICAgICAgIGNhcC5yZWxlYXNlKCkKICAgICAgICAgICAgZHVyYXRpb249ZGVjb2RlZC9mcHMKICAgICAgICAgICAg'
        'c3RhdHM9eyJkZWNvZGVkX2ZyYW1lcyI6ZGVjb2RlZCwiZGVjb2RlX3NlY29uZHMiOmRlY29kZV9zZWNvbmRzLAogICAgICAgICAg'
        'ICAgICAgICAgInNhbXBsZV9zdHJpZGUiOnN0cmlkZSwic2FtcGxlZF9mcmFtZXMiOmxlbihmcmFtZXNfdXNlZCksImRyb3BwZWRf'
        'ZnJhbWVzIjowfQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIG9yaWdpbj10aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAg'
        'IHN0YXRzPXJ1bl9yZXBsYXkoZW5naW5lLHZpZGVvLGVtaXQsbGFtYmRhIHQ6IGF1ZGlvLnRpY2sodCkgaWYgYXVkaW8gZWxzZSBO'
        'b25lLHNlY29uZHMsc3RhcnQ9b3JpZ2luKQogICAgICAgICAgICBkdXJhdGlvbj1zdGF0c1siZWxhcHNlZF9zIl0KICAgICAgICBp'
        'ZiBub3QgZnJhbWVzX3VzZWQ6IHJhaXNlIFJ1bnRpbWVFcnJvcigiVmlkZW8gcHJvZHVjZWQgbm8gdXNhYmxlIGZyYW1lcyIpCiAg'
        'ICAgICAgZXhwb3J0X2R1cmF0aW9uID0gYXVkaW8uZmluaXNoX2V4cG9ydChkdXJhdGlvbikgaWYgYXVkaW8gZWxzZSBkdXJhdGlv'
        'bgogICAgICAgIGlmIHByZXZpb3VzX2ltYWdlIGlzIG5vdCBOb25lOgogICAgICAgICAgICB3aGlsZSB2aWRlb19mcmFtZXM8bWF0'
        'aC5jZWlsKGV4cG9ydF9kdXJhdGlvbip3cml0ZXJfZnBzKToKICAgICAgICAgICAgICAgIHdyaXRlci53cml0ZShwcmV2aW91c19p'
        'bWFnZSk7dmlkZW9fZnJhbWVzKz0xCiAgICAgICAgaWYgd3JpdGVyOgogICAgICAgICAgICB3cml0ZXIucmVsZWFzZSgpO3dyaXRl'
        'cj1Ob25lCiAgICAgICAgaWYgYXVkaW86CiAgICAgICAgICAgIGF1ZGlvLnNhdmUob3V0cHV0LyJ2b2ljZS53YXYiLGV4cG9ydF9k'
        'dXJhdGlvbikKICAgICAgICBlbGFwc2VkPXRpbWUucGVyZl9jb3VudGVyKCktZWxhcHNlZF9zdGFydAogICAgICAgIHdpdGggKG91'
        'dHB1dC8iZGVjaXNpb25zLmpzb25sIikub3BlbigidyIsZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICAgICAgZm9yIHJl'
        'Y29yZCBpbiByZWNvcmRzOiBmLndyaXRlKGpzb24uZHVtcHMocmVjb3JkLGVuc3VyZV9hc2NpaT1GYWxzZSkrIlxuIikKICAgICAg'
        'ICB3aXRoIChvdXRwdXQvImF1ZGlvX2V2ZW50cy5qc29ubCIpLm9wZW4oInciLGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAg'
        'ICAgICAgIGZvciByZWNvcmQgaW4gYXVkaW8uZXZlbnRzIGlmIGF1ZGlvIGVsc2UgW106IGYud3JpdGUoanNvbi5kdW1wcyhyZWNv'
        'cmQsZW5zdXJlX2FzY2lpPUZhbHNlKSsiXG4iKQogICAgICAgIChvdXRwdXQvInNwZWVjaF9zZWdtZW50cy5qc29uIikud3JpdGVf'
        'dGV4dChqc29uLmR1bXBzKFsKICAgICAgICAgICAgeyJzdGFydCI6c1sic3RhcnQiXSwiZW5kIjpzWyJlbmQiXSwiYWN0aW9uIjpz'
        'WyJkZWNpc2lvbiJdLmFjdGlvbi52YWx1ZSwKICAgICAgICAgICAgICJldmlkZW5jZSI6c1siZGVjaXNpb24iXS5yZWNvcmQoKX0g'
        'Zm9yIHMgaW4gYXVkaW8uc2VnbWVudHNdIGlmIGF1ZGlvIGVsc2UgW10sCiAgICAgICAgICAgICBlbnN1cmVfYXNjaWk9RmFsc2Us'
        'aW5kZW50PTIpLGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgaW1wb3J0IGltYWdlaW9fZmZtcGVnCiAgICAgICAgY21kPVtpbWFn'
        'ZWlvX2ZmbXBlZy5nZXRfZmZtcGVnX2V4ZSgpLCIteSIsIi1sb2dsZXZlbCIsImVycm9yIiwiLWkiLHN0cihvdXRwdXQvInByZXZp'
        'ZXdfcmF3Lm1wNCIpXQogICAgICAgIGlmIGF1ZGlvOiBjbWQrPVsiLWkiLHN0cihvdXRwdXQvInZvaWNlLndhdiIpXQogICAgICAg'
        'IGNtZCs9WyItYzp2IiwibGlieDI2NCIsIi1waXhfZm10IiwieXV2NDIwcCIsIi1wcmVzZXQiLCJmYXN0IiwiLW1vdmZsYWdzIiwi'
        'K2Zhc3RzdGFydCJdCiAgICAgICAgaWYgYXVkaW86IGNtZCs9WyItYzphIiwiYWFjIiwiLXNob3J0ZXN0Il0KICAgICAgICBjbWQr'
        'PVtzdHIob3V0cHV0LyJkZW1vLm1wNCIpXQogICAgICAgIHN1YnByb2Nlc3MucnVuKGNtZCxjaGVjaz1UcnVlLGNhcHR1cmVfb3V0'
        'cHV0PVRydWUpCiAgICAgICAgdmVyc2lvbnM9e30KICAgICAgICBmb3IgbmFtZSBpbiAoInRvcmNoIiwidWx0cmFseXRpY3MiLCJ0'
        'cmFuc2Zvcm1lcnMiLCJvcGVuY3YtcHl0aG9uIiwibnVtcHkiLCJpbWFnZWlvLWZmbXBlZyIpOgogICAgICAgICAgICB2ZXJzaW9u'
        'c1tuYW1lXT1pbXBvcnRsaWIubWV0YWRhdGEudmVyc2lvbihuYW1lKQogICAgICAgIHJlcG9ydD17CiAgICAgICAgICAgICJzdGF0'
        'dXMiOiJ1bmxhYmVsbGVkX3Ntb2tlX29yX2RlbW9fbm90X2FjY2VwdGFuY2UiLCJtb2RlIjpjb25maWcubW9kZSwKICAgICAgICAg'
        'ICAgInZpZGVvIjpzdHIodmlkZW8pLCJ2aWRlb19zaGEyNTYiOmZpbmdlcnByaW50KHZpZGVvKSwidmlkZW9fZHVyYXRpb25fcyI6'
        'ZHVyYXRpb24sCiAgICAgICAgICAgICJleHBvcnRfZHVyYXRpb25fcyI6ZXhwb3J0X2R1cmF0aW9uLAogICAgICAgICAgICAic291'
        'cmNlX3ZpZGVvX2R1cmF0aW9uX3MiOnN0YXRzLmdldCgic291cmNlX2R1cmF0aW9uX3MiLGR1cmF0aW9uKSwKICAgICAgICAgICAg'
        'ImZyYW1lX3NlcXVlbmNlcyI6c29ydGVkKHNldChmcmFtZXNfdXNlZCkpLCJsb2FkX3NlY29uZHMiOmxvYWRfc2Vjb25kcywicHJv'
        'Y2Vzc2luZ193YWxsX3MiOmVsYXBzZWQsCiAgICAgICAgICAgICJwaXBlbGluZV9vdXRwdXRzX3Blcl9zZWNvbmQiOmxlbihyZWNv'
        'cmRzKS9tYXgoZWxhcHNlZCwxZS02KSwKICAgICAgICAgICAgInVuaXF1ZV9pbmZlcmVuY2VfZnJhbWVzX3Blcl9zZWNvbmQiOmxl'
        'bihzZXQoZnJhbWVzX3VzZWQpKS9tYXgoZWxhcHNlZCwxZS02KSwKICAgICAgICAgICAgImRlY29kZV9mcHMiOnN0YXRzWyJkZWNv'
        'ZGVkX2ZyYW1lcyJdL21heChzdGF0c1siZGVjb2RlX3NlY29uZHMiXSwxZS02KSwKICAgICAgICAgICAgInBpcGVsaW5lX21zIjpk'
        'aXN0cmlidXRpb24ocGlwZWxpbmVfbXMpLCJmcmFtZV90b19kZWNpc2lvbl9tcyI6ZGlzdHJpYnV0aW9uKHJlY2VpdmVkX2xhdGVu'
        'Y3kpLAogICAgICAgICAgICAibW9kZWxfbXMiOntrOmRpc3RyaWJ1dGlvbih2KSBmb3Igayx2IGluIGVuZ2luZS50aW1pbmdzLml0'
        'ZW1zKCl9LAogICAgICAgICAgICAibW9kZWxfZnBzIjp7azoxMDAwL25wLm1lYW4odikgaWYgdiBhbmQgbnAubWVhbih2KT4wIGVs'
        'c2UgTm9uZSBmb3Igayx2IGluIGVuZ2luZS50aW1pbmdzLml0ZW1zKCl9LAogICAgICAgICAgICAicnNzX3BlYWtfc2FtcGxlZF9t'
        'YiI6bWF4KHJzcyxkZWZhdWx0PTApLAogICAgICAgICAgICAiY3VkYV9wZWFrX2FsbG9jYXRlZF9tYiI6dG9yY2guY3VkYS5tYXhf'
        'bWVtb3J5X2FsbG9jYXRlZCgpLzEwMjQqKjIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlIE5vbmUsCiAgICAgICAg'
        'ICAgICJjdWRhX3BlYWtfcmVzZXJ2ZWRfbWIiOnRvcmNoLmN1ZGEubWF4X21lbW9yeV9yZXNlcnZlZCgpLzEwMjQqKjIgaWYgdG9y'
        'Y2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlIE5vbmUsCiAgICAgICAgICAgICJkZXZpY2VfbmFtZXMiOlt0b3JjaC5jdWRhLmdl'
        'dF9kZXZpY2VfbmFtZShpKSBmb3IgaSBpbiByYW5nZSh0b3JjaC5jdWRhLmRldmljZV9jb3VudCgpKV0sCiAgICAgICAgICAgICJ2'
        'ZXJzaW9ucyI6dmVyc2lvbnMsIm1vZGVsX3NoYTI1NiI6e20uaWQ6ZmluZ2VycHJpbnQobS5wYXRoKSBmb3IgbSBpbiBjb25maWcu'
        'bW9kZWxzIGlmIG0uZW5hYmxlZH0sCiAgICAgICAgICAgICJsb2FkX2ZhaWx1cmVzIjplbmdpbmUuZmFpbHVyZXMsImFjdGlvbl9j'
        'b3VudHMiOmRpY3QoY291bnRzKSwic291cmNlIjpzdGF0cywic3RhbGVfb3V0cHV0cyI6c3RhbGVfY291bnQsCiAgICAgICAgICAg'
        'ICJhdWRpbyI6eyJtb2RlIjoic2NoZWR1bGVkX2luX2V4cG9ydF9ub3RfaGFyZHdhcmVfcGxheWJhY2siIGlmIGF1ZGlvIGVsc2Ug'
        'ImRpc2FibGVkIiwKICAgICAgICAgICAgICAgICAgICAgInF1ZXVlX3dhaXRfbXMiOmRpc3RyaWJ1dGlvbihbZVsicXVldWVfd2Fp'
        'dF9zIl0qMTAwMCBmb3IgZSBpbiBhdWRpby5ldmVudHMgaWYgZVsiZXZlbnQiXT09InNjaGVkdWxlZCJdIGlmIGF1ZGlvIGVsc2Ug'
        'W10pLAogICAgICAgICAgICAgICAgICAgICAic3RhcnRlZF9ieV9hY3Rpb24iOmRpY3QoQ291bnRlcihlWyJhY3Rpb24iXSBmb3Ig'
        'ZSBpbiBhdWRpby5ldmVudHMgaWYgZVsiZXZlbnQiXT09InNjaGVkdWxlZCIpKSBpZiBhdWRpbyBlbHNlIHt9LAogICAgICAgICAg'
        'ICAgICAgICAgICAiaW50ZXJydXB0aW9uc19ieV9yZWFzb24iOmRpY3QoQ291bnRlcihlWyJyZWFzb24iXSBmb3IgZSBpbiBhdWRp'
        'by5ldmVudHMKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBpZiBlWyJl'
        'dmVudCJdPT0iY2FuY2VsbGVkIiBhbmQgZVsicmVhc29uIl0hPSJzdG9wcGVkX29yX211dGVkIikpIGlmIGF1ZGlvIGVsc2Uge30s'
        'CiAgICAgICAgICAgICAgICAgICAgICJkZXZpY2Vfc3VibWlzc2lvbl9tcyI6Tm9uZSwiaWRsZV9lbWVyZ2VuY3lfY2FwdHVyZV90'
        'b19kZXZpY2VfcDk1X21zIjpOb25lfSwKICAgICAgICAgICAgImFjY2VwdGFuY2UiOnsiZXZhbHVhdGVkIjpGYWxzZSwicmVhc29u'
        'IjoiUmVxdWlyZXMgaW5kZXBlbmRlbnQgZXZlbnQtbGFiZWxsZWQgdmlkZW9zLCA+PTEwMCBoYXphcmRzIGFuZCA+PTMwIGNsZWFy'
        'IG1pbnV0ZXMuIn0KICAgICAgICB9CiAgICAgICAgKG91dHB1dC8icmVwb3J0Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMo'
        'cmVwb3J0LGVuc3VyZV9hc2NpaT1GYWxzZSxpbmRlbnQ9MiksZW5jb2Rpbmc9InV0Zi04IikKICAgICAgICByZXR1cm4gcmVwb3J0'
        'CiAgICBmaW5hbGx5OgogICAgICAgIGlmIHdyaXRlcjogd3JpdGVyLnJlbGVhc2UoKQogICAgICAgIGVuZ2luZS5jbG9zZSgpCgpk'
        'ZWYgbWFpbigpOgogICAgcD1hcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcC5hZGRfYXJn'
        'dW1lbnQoIi0tY29uZmlnIixyZXF1aXJlZD1UcnVlKTtwLmFkZF9hcmd1bWVudCgiLS12aWRlbyIscmVxdWlyZWQ9VHJ1ZSkKICAg'
        'IHAuYWRkX2FyZ3VtZW50KCItLW91dHB1dCIscmVxdWlyZWQ9VHJ1ZSk7cC5hZGRfYXJndW1lbnQoIi0tc2Vjb25kcyIsdHlwZT1m'
        'bG9hdCkKICAgIHAuYWRkX2FyZ3VtZW50KCItLW1vZGUiLGNob2ljZXM9WyJvZmZsaW5lIiwicmVhbHRpbWVfcmVwbGF5Il0pCiAg'
        'ICBwLmFkZF9hcmd1bWVudCgiLS1lbnZpcm9ubWVudCIsY2hvaWNlcz1bImFsbGV5Iiwic2lkZXdhbGsiLCJpbmRvb3IiXSkKICAg'
        'IHAuYWRkX2FyZ3VtZW50KCItLW5vLWF1ZGlvIixhY3Rpb249InN0b3JlX3RydWUiKQogICAgYXJncz1wLnBhcnNlX2FyZ3MoKQog'
        'ICAgY29uZmlnPUNvbmZpZy5yZWFkKGFyZ3MuY29uZmlnKQogICAgaWYgYXJncy5tb2RlOiBjb25maWcubW9kZT1hcmdzLm1vZGUK'
        'ICAgIGlmIGFyZ3MuZW52aXJvbm1lbnQ6IGNvbmZpZy5lbnZpcm9ubWVudD1hcmdzLmVudmlyb25tZW50CiAgICBpZiBhcmdzLm5v'
        'X2F1ZGlvOiBjb25maWcuYXVkaW9fZW5hYmxlZD1GYWxzZQogICAgaWYgYXJncy5zZWNvbmRzIGlzIG5vdCBOb25lIGFuZCBhcmdz'
        'LnNlY29uZHM8PTA6IHAuZXJyb3IoIi0tc2Vjb25kcyBtdXN0IGJlIHBvc2l0aXZlIikKICAgIHJlcG9ydD1ydW4oY29uZmlnLGFy'
        'Z3MudmlkZW8sYXJncy5vdXRwdXQsYXJncy5zZWNvbmRzKQogICAgcHJpbnQoanNvbi5kdW1wcyh7Im91dHB1dCI6YXJncy5vdXRw'
        'dXQsImZwcyI6cmVwb3J0WyJ1bmlxdWVfaW5mZXJlbmNlX2ZyYW1lc19wZXJfc2Vjb25kIl0sImZhaWx1cmVzIjpyZXBvcnRbImxv'
        'YWRfZmFpbHVyZXMiXX0sZW5zdXJlX2FzY2lpPUZhbHNlKSkKaWYgX19uYW1lX189PSJfX21haW5fXyI6IG1haW4oKQoK'
        , 'cf09bcafe88f3cab5d2afacd51dd7400fb5f93617f3849199d44567d0c2929bf'
    ),
}
for name, (encoded, sha256) in speech_update.items():
    payload = b64decode(encoded)
    assert hashlib.sha256(payload).hexdigest() == sha256, name
    (WORK / name).write_bytes(payload)
print("Đã nạp bản đọc hết câu.")


## 2. Cài thư viện, giữ PyTorch CUDA của Kaggle

Chạy ô này trước khi import `transformers`/`ultralytics`. Nếu đã import phiên bản khác trong phiên hiện tại, restart session rồi chạy lại. Cấu hình này đã thử trên Windows CPU; ô kiểm tra tương thích dưới đây phải đạt trên Kaggle trước khi chạy video dài.


In [ ]:
import sys, subprocess, importlib.metadata as metadata
constraints = []
for package in ('torch', 'torchvision', 'torchaudio'):
    try: constraints.append(f'{package}=={metadata.version(package)}')
    except metadata.PackageNotFoundError: pass
Path('kaggle_torch_constraints.txt').write_text('\n'.join(constraints))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed',
                '-c', 'kaggle_torch_constraints.txt', '-r', 'requirements-guide-core.txt'], check=True)
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Bật GPU hoặc dừng để tránh nhầm benchmark CPU thành GPU.'
print('GPU:', torch.cuda.get_device_name(0))


## 3. Chọn video, bối cảnh và model

`alley`: hẻm/đường nhỏ; `sidewalk`: vỉa hè; `indoor`: hành lang trong nhà. `mirror_input=True` chỉ khi nguồn video **thực sự bị lật gương**. Không lật chỉ vì muốn ảnh trông giống camera selfie.

Mặc định một detector YOLOv8n ROD + SegFormer B0 ADE20K + Depth V2 Small. Đổi `PRIMARY` thành `rod_26` để dùng YOLO26n. `SECOND_DETECTOR=True` chỉ cho thử nghiệm so sánh, không mặc định tốt hơn.


In [ ]:
VIDEO_PATH = '/kaggle/input/datasets/nguyenduchng/guide-v2-train6/train6.mp4'
bundled_video = manifest.get('video')
assert VIDEO_PATH or bundled_video, 'Add Input video rồi điền VIDEO_PATH theo ô đầu.'
VIDEO = Path(VIDEO_PATH) if VIDEO_PATH else WORK / bundled_video
PRIMARY = 'rod_v8'  # 'rod_v8' hoặc 'rod_26'
SECOND_DETECTOR = False
ENVIRONMENT = 'indoor'
MODE = 'offline'  # 'realtime_replay' để đo bỏ khung/độ mới trên GPU
MAX_SECONDS = None
assert VIDEO.is_file(), f'Không thấy video: {VIDEO}'
assert PRIMARY in {'rod_v8', 'rod_26'}
config = json.loads(Path('configs/guide_kaggle.json').read_text())
config.update(environment=ENVIRONMENT, mode=MODE, mirror_input=False, sample_fps=10)
for m in config['models']:
    m['device'] = 'cuda:0'
    if m['role'] == 'detector': m['enabled'] = m['id'] == PRIMARY or SECOND_DETECTOR
Path('selected_config.json').write_text(json.dumps(config, indent=2))
print(json.dumps(config, indent=2))


## 4. Kiểm tra ba model bằng một khung hình

Ô này phải chạy xong không có `failures`. Không tiếp tục khi model chưa tải, không tương thích hoặc sai đường dẫn. Model depth trong gói đã được tải theo revision cố định.


In [ ]:
subprocess.run([sys.executable, 'tools/check_guide_environment.py', '--config', 'selected_config.json',
                '--video', str(VIDEO), '--report', '/kaggle/working/compatibility.json'], check=True)


## 5. Xuất video demo

Trong `offline`, lời cảnh báo được ghép theo thời gian video, **không phải độ trễ loa thực**. Trong `realtime_replay`, lõi mô phỏng nguồn camera chạy đúng thời gian, bỏ khung cũ và chỉ dùng kết quả còn mới. Notebook không đọc camera DroidCam qua mạng nội bộ.


In [ ]:
from datetime import datetime
RUN = Path('/kaggle/working') / ('guide_train6_finish_speech_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
command = [sys.executable, '-m', 'guide_core.demo', '--config', 'selected_config.json',
           '--video', str(VIDEO), '--output', str(RUN)]
if MAX_SECONDS is not None: command += ['--seconds', str(MAX_SECONDS)]
subprocess.run(command, check=True)
print('Đã xuất:', RUN/'demo.mp4')


In [ ]:
from IPython.display import Video, display
print((RUN/'report.json').read_text())
display(Video(str(RUN/'demo.mp4'), embed=True))


## 6. So sánh trên cùng khung hình (chủ động bật)

Chạy detector v8 / 26, thêm segmentation, thêm depth và thêm detector thứ hai. So sánh tốc độ này chưa đánh giá độ chính xác nếu chưa có nhãn sự kiện. Để thử 320/480/640 thêm `--sizes 320 480 640`. Mỗi thử nghiệm cần thời gian GPU; mặc định ô dưới **không chạy**.


In [ ]:
RUN_COMPARISON = False
if RUN_COMPARISON:
    compare_dir = Path('/kaggle/working')/('guide_compare_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
    subprocess.run([sys.executable, 'tools/compare_guide_models.py', '--config', 'selected_config.json',
                    '--video', str(VIDEO), '--output', str(compare_dir), '--seconds', '6', '--sizes', '480'], check=True)
    print((compare_dir/'comparison.json').read_text())


## 7. Chấm nhãn và lưu đầu ra

Trên laptop mở `tools/annotate_guide.html` để đánh dấu khoảng nguy hiểm, thông thoáng, hướng hợp lệ và hướng bị cấm. Tải JSON sau khi kiểm tra nhãn. Khi chấm, đặt `MAX_SECONDS = None` để chạy hết video. Dùng `python -m guide_core.evaluate --annotations <file.json> --run <RUN>` cho một video. Dùng `python -m guide_core.evaluate --manifest <manifest.json>` để gộp nhiều video cùng tập validation hoặc test.

Đầu ra cần tải: `demo.mp4`, `report.json`, `decisions.jsonl`, `audio_events.jsonl`, `speech_segments.json`. Save Version sau khi chạy hoàn tất. Chọn cách lưu không chạy lại nếu chỉ muốn giữ output phiên hiện tại. Không coi có file video là đã đạt mục tiêu an toàn.


In [ ]:
segments = json.loads((RUN / 'speech_segments.json').read_text(encoding='utf-8'))
events = [json.loads(line) for line in (RUN / 'audio_events.jsonl').read_text(encoding='utf-8').splitlines()]
overlaps = sum(a['end'] > b['start'] + 1e-6 for a, b in zip(segments, segments[1:]))
interruptions = sum(event['event'] == 'cancelled' for event in events)
assert (overlaps, interruptions) == (0, 0), (overlaps, interruptions)
print(f'{len(segments)} đoạn lời, {overlaps} chồng tiếng, {interruptions} ngắt giữa câu.')
archive = shutil.make_archive(str(RUN), 'zip', RUN)
print('Tải tại Output:', archive)
